In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/reference.csv
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/AIMO3_Reference_Problems.pdf
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/sample_submission.csv
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/test.csv
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/aimo_3_inference_server.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/aimo_3_gateway.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/__init__.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/core/templates.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/core/base_gateway.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/core/relay.py
/kaggle/input/comp

In [2]:
import os
import sys
import subprocess
import time
import threading
import signal
import json
import shutil
import gc
import re
import atexit
import queue
import ast
import textwrap
import concurrent.futures
import site
import ctypes
from datetime import datetime
from typing import List, Dict, Optional, Any, Union
from collections import Counter
from dataclasses import dataclass, field

# Data processing
try:
    import pandas as pd
    import polars as pl
    import numpy as np
    import torch
    torch.set_default_dtype(torch.bfloat16)
except ImportError:
    pass

import jupyter_client

# ==========================================
# 🛠️ GLOBAL CONFIGURATION & RESOURCE LOCKS
# ==========================================
KAGGLE_MODE = os.path.exists('/kaggle/working') or os.path.exists('/kaggle/input')
run_type = os.getenv('KAGGLE_KERNEL_RUN_TYPE', 'Local')
VERBOSE_MODE = True
MAX_KAGGLE_SECONDS = 32400 # 9 Hours
KAGGLE_SUBMISSION_RECORDS = [] # 🔥 GLOBAL TRACKING FOR SUBMISSION FAILSAFE
START_TIME = time.time()
EXPERIMENTAL_MODE = False # 🧪 Set to True to run local resource diagnostics vs Kaggle Server

# 🚨 SYSTEM SILENCE & STABILITY FLAGS
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["VLLM_LOGGING_LEVEL"] = "WARNING"
os.environ["NCCL_LOG_LEVEL"] = "ERROR"

def vprint(*args, **kwargs):
    print(*args, **kwargs, flush=True)

# --- PROTOBUF PATCH MOVED TO BOOTSTRAP ---

# ==========================================
# 📦 BOOTSTRAP: ENVIRONMENT SYNCHRONIZATION
# ==========================================
def bootstrap_kaggle_offline():
    if not os.path.exists('/kaggle/input'): return
    
    # 🕵️  PATH PURIFICATION: Force site.getusersitepackages() to the front
    # This ensures our synced wheels override protected system packages
    try:
        user_site = site.getusersitepackages()
        if user_site not in sys.path:
            sys.path.insert(0, user_site)
        os.environ["PYTHONPATH"] = f"{user_site}:{os.environ.get('PYTHONPATH', '')}"
    except:
        user_site = os.path.expanduser("~/.local/lib/python3.12/site-packages")
        if user_site not in sys.path: sys.path.insert(0, user_site)
    
    vprint("\n" + "="*50)
    vprint("🚀 [BOOTSTRAP]: Offline environment synchronization active.")
    vprint(f"📍 [PATH]: Prioritizing {user_site}")
    vprint("="*50)
    
    # 🕵️  Internet Check: Prevent pip-hang in offline kernels
    import socket
    has_internet = False
    try:
        socket.setdefaulttimeout(1)
        socket.socket(socket.AF_INET, socket.SOCK_STREAM).connect(("8.8.8.8", 53))
        has_internet = True
        vprint("🌐 [BOOTSTRAP]: Online connection detected.")
    except:
        vprint("📡 [BOOTSTRAP]: Offline mode confirmed.")

    # 🛡️  UNIVERSAL PROTOBUF BRIDGE (Sticky Stability Fix)
    try:
        import google.protobuf.message_factory as mf
        from google.protobuf import symbol_database
        def patch_factory(factory_instance):
            if not hasattr(factory_instance, 'GetPrototype'):
                factory_instance.GetPrototype = lambda desc: symbol_database.Default().GetSymbol(desc.full_name)
        _original_init = mf.MessageFactory.__init__
        def _patched_init(self, *args, **kwargs):
            _original_init(self, *args, **kwargs)
            patch_factory(self)
        mf.MessageFactory.__init__ = _patched_init
        vprint("🧱 [PATCH]: Sticky Protobuf bridge established.")
    except: pass

    VLLM_WHEELS = "/kaggle/input/datasets/liquidvisualsinteractive/vllm-wheels/vllm-wheels/"
    
    vprint("🕵️  Verifying vLLM availability...")
    need_sync = False
    try:
        import vllm
        from vllm import LLM, SamplingParams
        vprint(f"✅ vLLM {vllm.__version__} is already available and verified.")
    except (ImportError, AttributeError, RuntimeError) as e:
        vprint(f"📦 [BOOTSTRAP]: vLLM issues detected: {e}")
        need_sync = True

    if need_sync and os.path.exists(VLLM_WHEELS):
        vprint("🚀 [SYNC]: Initializing AGGRESSIVE Environment Repair Sequence...")
        import glob
        find_links = f"--find-links={VLLM_WHEELS}"
        all_whls = sorted(glob.glob(os.path.join(VLLM_WHEELS, "**", "*.whl"), recursive=True))
        
        # 🔨 Step A: Force-uninstall existing torch/vllm to ensure no stale symbols remain
        vprint("🗑️  [SYNC]: Purging conflicting pre-installed binaries...")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torch", "vllm", "xformers", "transformers"], 
                       check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        
        # 🔨 Step B: Install in STRICT SEQUENCE (Torch must be first)
        categories = ["torch-", "xformers-", "vllm-", "transformers-"]
        for cat in categories:
            match = [w for w in all_whls if cat in os.path.basename(w).lower()]
            for whl_file in match:
                vprint(f"   ⚙️  [CORE SYNC]: {os.path.basename(whl_file)}...")
                subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", find_links, "--force-reinstall", whl_file], 
                               check=False)

        # 🔨 Step C: Install everything else
        total_remaining = [w for w in all_whls if not any(c in os.path.basename(w).lower() for c in categories) and not any(x in os.path.basename(w).lower() for x in ["protobuf", "pillow", "numpy"])]
        count = 0
        total = len(total_remaining)
        
        for whl_file in total_remaining:
            count += 1
            basename = os.path.basename(whl_file)
            vprint(f"   📦 [SYNC {count}/{total}]: {basename}...")
            subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", find_links, "--force-reinstall", whl_file], 
                           check=False)
        
        import importlib
        importlib.invalidate_caches()
        
        # 🛡️  MODULE PURGE: Clear any system-cached versions of critical libs
        # This prevents Python from using the stale /usr/local/ symbols
        for m in ['vllm', 'torch', 'transformers', 'xformers', 'pydantic']:
            if m in sys.modules:
                del sys.modules[m]

        # 🧪 [STEEL-LINKER V2]: Manual symbol binding from verified local path
        try:
            user_site = site.getusersitepackages()
            local_torch_lib = os.path.join(user_site, "torch", "lib")
            if not os.path.exists(local_torch_lib):
                # Fallback for different Kaggle environment structures
                local_torch_lib = os.path.expanduser("~/.local/lib/python3.12/site-packages/torch/lib")
            
            if os.path.exists(local_torch_lib):
                libs_to_load = ["libtorch_cuda.so", "libc10_cuda.so"]
                for lib in libs_to_load:
                    lp = os.path.join(local_torch_lib, lib)
                    if os.path.exists(lp):
                        vprint(f"🔗 [STEEL-LINKER]: Pre-binding {lib} from local site...")
                        ctypes.CDLL(lp, mode=ctypes.RTLD_GLOBAL)
            else:
                vprint("⚠️ [STEEL-LINKER]: Local torch lib not found for pre-binding.")
        except Exception as e:
            vprint(f"⚠️ [STEEL-LINKER]: Pre-bind warning: {e}")

        vprint("✅ [SYNC]: Environment synchronized and purified.")

    elif not os.path.exists(VLLM_WHEELS):
        vprint("⚠️ [BOOTSTRAP]: Wheel dataset not found. Skipping sync.")

    # 🕵️  DEEP PATH AUDIT: Log exactly where things are loading from
    vprint("\n" + "-"*50)
    vprint("🕵️  [DEEP PATH AUDIT]: Verifying Library Lineage...")
    try:
        import torch
        import vllm
        import transformers
        vprint(f"   📍 TORCH: {torch.__version__} @ {torch.__file__}")
        vprint(f"   📍 vLLM: {vllm.__version__} @ {vllm.__file__}")
        vprint(f"   📍 TRANS: {transformers.__version__} @ {transformers.__file__}")
    except Exception as e:
        vprint(f"   ⚠️  Audit partial failure: {e}")
    vprint("-"*50 + "\n")

    vprint("✅ Bootstrap Complete.\n" + "="*50 + "\n")

def _fix_kaggle_weight_filenames(model_path: str) -> str:
    if not KAGGLE_MODE: return model_path
    index_path = os.path.join(model_path, "model.safetensors.index.json")
    if not os.path.isfile(index_path): return model_path
    fixed_path = os.path.join("/kaggle/working", "fixed_weights", os.path.basename(model_path.rstrip("/")))
    if os.path.exists(fixed_path): return fixed_path
    os.makedirs(fixed_path, exist_ok=True)
    for item in os.listdir(model_path):
        src, dst = os.path.join(model_path, item), os.path.join(fixed_path, item)
        if os.path.isfile(src):
            try: os.symlink(src, dst)
            except: shutil.copy(src, dst)
    return fixed_path

# ==========================================
# 🧠 INTELLECT: ModernBERT & IntelLib
# ==========================================
class ModernBertEngine:
    def __init__(self, model_search_paths: List[str] = None):
        self.device = "cpu"
        self.model = None
        self.tokenizer = None
        self.is_active = False
        self.intel_cache = {}

        paths = model_search_paths or ["/kaggle/input/datasets/liquidvisualsinteractive/modernbert-base/"]
        model_path = next((p for p in paths if os.path.exists(os.path.join(p, "config.json"))), None)

        if model_path:
            try:
                from transformers import AutoTokenizer, AutoModelForMaskedLM, AutoConfig
                self.tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
                
                # Resilient Discovery: Try specific then generic
                model_class = None
                try:
                    from transformers import ModernBertForMaskedLM, ModernBertModel
                except: pass
                
                import transformers
                for target in ["ModernBertForMaskedLM", "ModernBertModel", "AutoModelForMaskedLM"]:
                    if hasattr(transformers, target):
                        model_class = getattr(transformers, target)
                        break
                
                if model_class:
                    vprint(f"🧠 [MODERNBERT]: Loading via {model_class.__name__}...")
                    self.model = model_class.from_pretrained(model_path, local_files_only=True, torch_dtype=torch.bfloat16).to(self.device).eval()
                else:
                    vprint("⚠️ [MODERNBERT]: Could not discover suitable model class.")
                
                vprint(f"✅ [MODERNBERT]: Model loaded successfully on {self.device}")
                self.is_active = True
                self.cache_intel()
            except Exception as e:
                vprint(f"⚠️ [MODERNBERT ERROR]: {e}")

    def cache_intel(self):
        intel_path = "/kaggle/input/datasets/liquidvisualsinteractive/maa-14b-core-intellib/"
        if not os.path.exists(intel_path): return
        sys.stdout.write("🧠 [INTELLIB]: Buffering Math Archives... ")
        sys.stdout.flush()
        for f in ['theorems.parquet', 'axioms.parquet', 'strategies.parquet', 'definitions.parquet']:
            p = os.path.join(intel_path, f)
            if os.path.exists(p):
                try: self.intel_cache[f] = pd.read_parquet(p)
                except: pass
        print(" ✅ [CACHED]")

    def classify(self, text: str) -> str:
        return "Algebra"

# ==========================================
# 📜 SYMBOLIC: Jupyter Sandbox & Z3
# ==========================================
class JupyterSandbox:
    def __init__(self, kernel_manager, kernel_client):
        self.km = kernel_manager
        self.kc = kernel_client
        self.timeout = 10 
        self.execute("try: import resource; resource.setrlimit(resource.RLIMIT_AS, (8000000000, 8000000000))\nexcept Exception: pass")

    def execute(self, code: str) -> tuple[str, str, int, Any]:
        self.kc.execute(code)
        stdout, stderr, exec_res, return_code = "", "", None, 0
        try:
            while True:
                msg = self.kc.get_iopub_msg(timeout=self.timeout)
                msg_type = msg['header']['msg_type']
                if msg_type == 'stream':
                    stdout += msg['content']['text'] if msg['content']['name'] == 'stdout' else ""
                elif msg_type == 'execute_result':
                    exec_res = msg['content']['data'].get('text/plain', None)
                elif msg_type == 'error':
                    stderr += "\n".join(msg['content']['traceback'])
                    return_code = 1
                elif msg_type == 'status' and msg['content']['execution_state'] == 'idle':
                    break
        except queue.Empty:
            return "", "Timeout", 124, None
        return stdout.strip(), stderr.strip(), return_code, exec_res

class KernelPoolManager:
    def __init__(self, pool_size: int = 2):
        self.pool = queue.Queue()
        for _ in range(pool_size):
            km = jupyter_client.KernelManager(kernel_name='python3', extra_arguments=['--HistoryManager.enabled=False'])
            km.start_kernel()
            kc = km.client()
            kc.start_channels()
            self.pool.put(JupyterSandbox(km, kc))
        atexit.register(self.cleanup)

    def get_sandbox(self) -> JupyterSandbox: return self.pool.get()
    def cleanup(self):
        vprint("🧼 [CLEANUP]: Sandbox Pool Drained.")

# ==========================================
# 🚀 NEURAL: 14B vLLM Orchestrator
# ==========================================
@dataclass
class SessionMemory:
    solved_ids: List[str] = field(default_factory=list)

class ModelOrchestrator:
    def __init__(self, model_path, kernel_pool, start_time, session_memory):
        self.model_path = model_path
        self.kernel_pool = kernel_pool
        self.llm = None
        self._initialize_llm(model_path)

    def _initialize_llm(self, model_path):
        if model_path == "mock":
            vprint("🧪 [LLM]: Initializing Mock Engine.")
            return
        
        fixed_path = _fix_kaggle_weight_filenames(model_path)
        try:
            from vllm import LLM
            self.llm = LLM(
                model=fixed_path,
                trust_remote_code=True,
                gpu_memory_utilization=0.91,
                max_model_len=16384,
                dtype="bfloat16",
                enforce_eager=True,
                disable_log_stats=True
            )
            vprint(f"✅ [LLM]: 14B Model loaded from {os.path.basename(fixed_path)}")
        except Exception as e:
            vprint(f"🧨 [LLM ERROR]: {e}")

    def predict(self, prompt: str) -> str:
        """🚀 Neural Inference: Basic generation for diagnostic testing."""
        if self.llm is None:
            return "Mock Response: vLLM is not loaded."
        
        try:
            from vllm import SamplingParams
            params = SamplingParams(temperature=0, max_tokens=100)
            outputs = self.llm.generate([prompt], params)
            return outputs[0].outputs[0].text
        except Exception as e:
            return f"Inference Error: {e}"

class Model:
    """🎬 Submission Wrapper: Facilitates predictable handshake between Server and Orchestrator."""
    def __init__(self):
        self.orchestrator = None
        self.brain = None

    def load(self):
        """🚀 THE PERFECT SEQUENCE: Hierarchical Resource Initialization."""
        if self.orchestrator is not None:
            return

        vprint("\n" + "⚙️"*35)
        vprint("🚀 [BOOT]: STARTING HIERARCHICAL RESOURCE LOADING SEQUENCE")
        vprint("⚙️"*35 + "\n")

        # --- STEP 1: SYMBOLIC LAYER (Sandbox/Z3) ---
        # We start kernels while the system is "cold" and RAM is clear.
        try:
            vprint("🏁 [LOAD 1/3]: Initializing Symbolic Sandbox (Jupyter/Z3)...")
            pool = KernelPoolManager(pool_size=1)
            vprint("✅ [SANDBOX]: Kernels spawned and Z3 ready.")
        except Exception as e:
            vprint(f"❌ [CRITICAL ERROR]: Sandbox ignition failed: {e}")
            raise

        # --- STEP 2: INTELLECTUAL LAYER (ModernBERT/IntelLib) ---
        # CPU-bound retrieval logic warmed up before LLM claims VRAM.
        try:
            vprint("🏁 [LOAD 2/3]: Initializing Intellectual Retrieval (ModernBERT)...")
            self.brain = ModernBertEngine()
            vprint("✅ [BRAIN]: IntelLib buffered and ModernBERT ready.")
        except Exception as e:
            vprint(f"⚠️ [NON-CRITICAL]: Intellectual layer failed: {e}")

        # --- STEP 3: NEURAL LAYER (14B vLLM Engine) ---
        # Final intensive VRAM allocation for the 14B model.
        try:
            vprint("🏁 [LOAD 3/3]: Initializing Neural Heavyweight (14B vLLM)...")
            model_paths = [
                "/kaggle/input/datasets/liquidvisualsinteractive/maa-deepseek-qwen-14b-ties-merged/",
                "/kaggle/input/datasets/avergonzado/maa-deepseek-qwen-14b-ties-merged/"
            ]
            model_path = next((p for p in model_paths if os.path.exists(os.path.join(p, "config.json"))), None)
            
            if not model_path:
                vprint("⚠️ [NEURAL]: Real model not found. Falling back to Mock Engine for flow validation.")
                model_path = "mock"
            
            self.orchestrator = ModelOrchestrator(model_path, pool, START_TIME, SessionMemory())
            vprint("✅ [NEURAL]: 14B Model Orchestrator live.")
        except Exception as e:
            vprint(f"❌ [CRITICAL ERROR]: Neural engine ignition failed: {e}")
            raise

        vprint("\n" + "🏁"*35)
        vprint("✅ [SEQUENCE COMPLETE]: ALL SYSTEMS MISSION-READY")
        vprint("🏁"*35 + "\n")

    def solve(self, problem_text: str) -> int:
        """
        🧠 [LOGIC INJECTION POINT]
        Interface for your mathematical solving logic.
        """
        # 📡 [LOGIC HEARTBEAT]: Confirming receipt of data from Kaggle
        vprint(f"🧠 [SOLVER]: Received problem (Length: {len(problem_text)}).")
        vprint(f"🧠 [SOLVER]: Executing dummy logic... returning 0.")

        if self.orchestrator is None:
            self.load()
            
        # TODO: PLUG YOUR ACTUAL SOLVER LOGIC HERE
        # Output should be an integer between 0 and 999 999.
        final_answer = 0 
        
        return final_answer

# Global singleton for the submission harness
model = Model()

def predict(id_: pl.Series, problem: pl.Series) -> pl.DataFrame:
    """
    🏁 KAGGLER API GATEWAY.
    Interfaces with the AIMO 3 Inference Server.
    """
    # 1. Unpack Request
    id_val = id_.item(0) if hasattr(id_, "item") else id_[0]
    problem_text = problem.item(0) if hasattr(problem, "item") else problem[0]
    
    # 📡 [SERVICE HEARTBEAT]: Log arrival of new problem
    vprint(f"\n{'='*70}\n📥 [REQUEST RECEIVED]: ID {id_val} | Time: {datetime.now().strftime('%H:%M:%S')}\n{'='*70}")

    # 🛑 [ORBITAL DECAY GUARD]: 4.8 Hour Hard Limit
    elapsed = time.time() - START_TIME
    if elapsed > 17280: # 4.8 Hours
        vprint(f"🚨 [PANIC] Budget exhausted! Returning default safety answer 0 for ID {id_val}...")
        return pl.DataFrame({'id': id_, 'answer': pl.Series([0], dtype=pl.Int64)})

    vprint(f"\n\n[ID {id_val}] Solving...")
    
    # Reasoning Thinking Timer (for visibility in Kaggle logs)
    stop_h = threading.Event()
    def thinking_fn():
        t0 = time.time()
        while not stop_h.is_set():
            sys.stdout.write(f"\r🤔 Thinking... [ {int(time.time() - t0)} s ]")
            sys.stdout.flush()
            time.sleep(1)
    
    h_thread = threading.Thread(target=thinking_fn)
    h_thread.daemon = True
    h_thread.start()
    
    try:
        # 🧠 [CONNECT TO LOGIC]: Passing the problem to our logic layer
        final_val = model.solve(problem_text)
        
        # 🛡️ THE FAILSAFE: Track manually
        KAGGLE_SUBMISSION_RECORDS.append({'id': id_val, 'answer': final_val})
    except Exception as e:
        vprint(f"❌ [GATEWAY ERROR]: Unexpected failure in predict: {e}")
        final_val = 0
    finally:
        stop_h.set()
        h_thread.join()
        print()
    
    vprint(f"✅ Verified Result: {final_val} | ID: {id_val}")
    
    # 🛠️ THE SCHEMA FIX: Strict Polars matching
    return pl.DataFrame({
        'id': id_, 
        'answer': pl.Series([final_val], dtype=pl.Int64)
    })

# ==========================================
# 🏁 FOUNDATION MAIN: System Diagnostic
# ==========================================
def run_diagnostic_suite():
    """🧪 System Stress Test: Sequentially verifies all engine layers."""
    vprint("\n" + "!"*70)
    vprint("☢️ [EXPERIMENTAL MODE]: RUNNING INDEPENDENT DIAGNOSTIC SUITE")
    vprint("!"*70 + "\n")
    
    # 🔥 SINGLE POINT OF FAILURE BOOT:
    model.load()
    
    # 1. Intellectual Layer (ModernBERT)
    vprint("\n🧠 [DIAGNOSTIC 1/3]: Testing Intellectual Retrieval (ModernBERT)...")
    if model.brain:
        category = model.brain.classify("Solve the equation x+2=4")
        vprint(f"✅ [BERT]: Classification Result: {category}")
    else:
        vprint("⚠️ [BERT]: Skip - Layer not initialized.")
    
    # 2. Symbolic Layer (Sandbox/Z3)
    vprint("\n📜 [DIAGNOSTIC 2/3]: Testing Symbolic Execution (Sandbox/Z3)...")
    if model.orchestrator and model.orchestrator.kernel_pool:
        sandbox = model.orchestrator.kernel_pool.get_sandbox()
        stdout, stderr, r_code, res = sandbox.execute("import z3; x = z3.Int('x'); print(z3.solve(x + 2 == 4))")
        if r_code == 0:
            vprint(f"✅ [SANDBOX]: Symbolic Match! Output: {stdout}")
        else:
            vprint(f"❌ [SANDBOX]: Execution Failed: {stderr}")
    else:
        vprint("❌ [SANDBOX]: Skip - Pool not available.")
    
    # 3. Neural Layer (14B vLLM)
    vprint("\n🚀 [DIAGNOSTIC 3/3]: Testing Neural Generation (14B vLLM)...")
    if model.orchestrator:
        test_response = model.orchestrator.predict("Complete this sequence: 1, 2, 3,")
        vprint(f"✅ [LLM]: Neural Response: {test_response.strip()}")
    else:
        vprint("❌ [LLM]: Skip - Orchestrator failed to initialize.")
    
    vprint("\n" + "="*70)
    vprint("🏁 [DIAGNOSTIC COMPLETE]: All systems reporting OK.")
    vprint("="*70 + "\n")

if __name__ == "__main__":
    bootstrap_kaggle_offline()
    
    # 🏁 HIERARCHICAL BOOT (Always required for neural readiness)
    vprint("\n🚀 [SUBMISSION]: Preparing AIMO Foundation Skeleton...")
    try:
        model.load()
    except Exception as e:
        vprint(f"❌ [BOOT ERROR]: Heavy resource initialization failed: {e}")
            
    if EXPERIMENTAL_MODE:
        run_diagnostic_suite()
        sys.exit(0)

    if KAGGLE_MODE:
            
        import kaggle_evaluation.aimo_3_inference_server
        vprint("📡 [SERVER]: Initializing AIMO 3 Inference Server...")
        server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)
        
        vprint("\n" + "="*70)
        vprint("[READY]: Inference Server is now listening. Standing by for competition problems...")
        vprint("=" * 70 + "\n")
        
        try:
            server.serve()
        except Exception as e:
            vprint(f"❌ [SERVER CRASH]: API threw an error during serve(): {e}")
            
        # --- 🛡️ THE MANUAL OVERRIDE (Failsafe) ---
        # If the API container hangs or fails to write the file, we write the tracked records.
        if KAGGLE_SUBMISSION_RECORDS:
            try:
                pl.DataFrame(KAGGLE_SUBMISSION_RECORDS, schema={'id': pl.Utf8, 'answer': pl.Int64}).write_parquet('submission.parquet')
                vprint(f"✅ [FAILSAFE]: Successfully saved {len(KAGGLE_SUBMISSION_RECORDS)} manual records to submission.parquet")
            except Exception as e:
                vprint(f"❌ [FAILSAFE ERROR]: Failed to write manual parquet: {e}")

    else:
        vprint("🧪 [DEVELOPMENT]: Running local diagnostic main.")
        # Replace with your local testing logic as needed
        vprint("Foundation is ready. No competition environment detected.")



🚀 [BOOTSTRAP]: Offline environment synchronization active.
📍 [PATH]: Prioritizing /root/.local/lib/python3.12/site-packages
📡 [BOOTSTRAP]: Offline mode confirmed.
🧱 [PATCH]: Sticky Protobuf bridge established.
🕵️  Verifying vLLM availability...
📦 [BOOTSTRAP]: vLLM issues detected: No module named 'vllm'
🚀 [SYNC]: Initializing AGGRESSIVE Environment Repair Sequence...
🗑️  [SYNC]: Purging conflicting pre-installed binaries...
   ⚙️  [CORE SYNC]: torch-2.9.1-cp312-cp312-manylinux_2_28_x86_64.whl...
Looking in links: /kaggle/input/datasets/liquidvisualsinteractive/vllm-wheels/vllm-wheels/
Processing /kaggle/input/datasets/liquidvisualsinteractive/vllm-wheels/vllm-wheels/torch-2.9.1-cp312-cp312-manylinux_2_28_x86_64.whl
   ⚙️  [CORE SYNC]: vllm-0.16.0-cp38-abi3-manylinux_2_31_x86_64.whl...
Looking in links: /kaggle/input/datasets/liquidvisualsinteractive/vllm-wheels/vllm-wheels/
Processing /kaggle/input/datasets/liquidvisualsinteractive/vllm-wheels/vllm-wheels/vllm-0.16.0-cp38-abi3-manylin

0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.


✅ [SANDBOX]: Kernels spawned and Z3 ready.
🏁 [LOAD 2/3]: Initializing Intellectual Retrieval (ModernBERT)...
⚠️ [MODERNBERT ERROR]: Only a single TORCH_LIBRARY can be used to register the namespace triton; please put all of your definitions in a single TORCH_LIBRARY block.  If you were trying to specify implementations, consider using TORCH_LIBRARY_IMPL (which can be duplicated).  If you really intended to define operators for a single namespace in a distributed way, you can use TORCH_LIBRARY_FRAGMENT to explicitly indicate this.  Previous registration of TORCH_LIBRARY was registered at /usr/local/lib/python3.12/dist-packages/torch/__init__.py:2805; latest registration was registered at /usr/local/lib/python3.12/dist-packages/torch/__init__.py:2700
✅ [BRAIN]: IntelLib buffered and ModernBERT ready.
🏁 [LOAD 3/3]: Initializing Neural Heavyweight (14B vLLM)...
🧨 [LLM ERROR]: Only a single TORCH_LIBRARY can be used to register the namespace triton; please put all of your definitions in a s